In [1]:
#Import Required Libraries
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
 #Load Data
data = pd.read_csv("spam.csv", encoding="latin-1")
data = data[['Category', 'Message']]

print("Dataset Shape:", data.shape)
print("\nClass Distribution:")
print(data['Category'].value_counts())

Dataset Shape: (5572, 2)

Class Distribution:
Category
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
  # Clean Data
data = data.drop_duplicates()
data = data.dropna()

# Convert labels to numeric
data['label'] = data['Category'].map({'ham': 0, 'spam': 1})

In [8]:

#  Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

data['clean_message'] = data['Message'].apply(clean_text)

In [9]:
#  Split Data
X = data['clean_message']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
# Feature Engineering
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.95,
    min_df=2,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [11]:
# Build Model
model = MultinomialNB()

#  Train Model
model.fit(X_train_vec, y_train)

#  Predictions
predictions = model.predict(X_test_vec)

# Evaluation
accuracy = accuracy_score(y_test, predictions)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=['Ham', 'Spam']
))



Accuracy: 0.9632

Confusion Matrix:
[[904   0]
 [ 38  90]]

Classification Report:
              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       904
        Spam       1.00      0.70      0.83       128

    accuracy                           0.96      1032
   macro avg       0.98      0.85      0.90      1032
weighted avg       0.96      0.96      0.96      1032



In [12]:
#  Test New Messages
new_messages = [
    "Congratulations! You have won a free iPhone.",
    "Hi, are we still meeting tomorrow?"
]

new_messages_vec = vectorizer.transform(new_messages)
results = model.predict(new_messages_vec)

print("\nNew Message Predictions:")
for msg, pred in zip(new_messages, results):
    label = "Spam" if pred == 1 else "Ham"
    print(f"{label}: {msg}")


New Message Predictions:
Spam: Congratulations! You have won a free iPhone.
Ham: Hi, are we still meeting tomorrow?
